# Day 4 — Building with LLM APIs

## What we're building
A Sleep Research Assistant powered by Claude API that:
1. Answers sleep science questions with citations
2. Analyses experiment results and gives feedback  
3. Suggests next steps for Project 07
4. Demonstrates multi-turn conversation management

## Why this matters
Every serious AI application today is built on top of
foundation models via API. Learning to build with LLMs
is as fundamental as learning to use databases was in 2000.

## Key concepts
- API calls and authentication
- Prompt engineering
- Conversation history management
- System prompts and personas
- Structured output extraction

In [1]:
!pip install anthropic

In [2]:
import anthropic
import json
import textwrap
from datetime import datetime

# Install if needed:
# !pip install anthropic

# ── Initialise client ─────────────────────────────────────
# API key is handled automatically in Claude.ai environment
client = anthropic.Anthropic()

print("Anthropic client ready")
print(f"   Using model: claude-sonnet-4-6")

Anthropic client ready
   Using model: claude-sonnet-4-6


In [8]:
import json
import textwrap
from datetime import datetime

# ── Mock LLM client ───────────────────────────────────────
# Simulates API calls with pre-written responses
# Replace with real client when you have an API key

class MockLLMClient:
    """
    Simulates an LLM API for learning purposes.
    Same interface as real client — swap one line to go live.
    """

    def __init__(self):
        self.call_count = 0
        print("MockLLMClient ready")
        print("(Replace with anthropic.Anthropic() "
              "when you have an API key)\n")

    def ask(self, prompt, system=None, max_tokens=500):
        self.call_count += 1

        # Simulate different responses based on keywords
        p = prompt.lower()

        if 'n1' in p and 'hard' in p:
            return """N1 is the hardest sleep stage to classify for three reasons:

1. BREVITY: N1 typically lasts only 1-7 minutes, giving the 
   classifier very few training examples relative to N2 (45-55%).

2. SPECTRAL OVERLAP: N1 EEG shows mixed frequencies — alpha 
   waves (8-13Hz) from wake begin to slow into theta (4-8Hz). 
   This overlap makes it ambiguous.

3. INTER-SUBJECT VARIABILITY: N1 signatures vary more between 
   people than any other stage, making generalisation harder.

For your CNN-LSTM (N1 F1=0.47), the CNN windows help because 
they capture the TRANSITION from alpha to theta across the 
30-second epoch — exactly what distinguishes N1 from wake."""

        elif 'rem' in p and ('feature' in p or 'eeg' in p):
            return """Most discriminative EEG features for REM vs NREM:

1. THETA POWER (4-8Hz): strongly elevated in REM
2. THETA/ALPHA RATIO: peaks during REM, low in NREM
3. SAWTOOTH WAVES: 2-6Hz bursts, nearly unique to REM
4. LOW DELTA POWER: REM has less delta than N2/N3
5. SPINDLE ABSENCE: sleep spindles (12-15Hz) disappear in REM

For Project 07: your theta/alpha ratio feature in 
extract_features() is the single most important feature.
If you haven't added it yet, add it — it should give 
a clear accuracy boost in detect_rem.py."""

        elif 'next' in p or 'improve' in p or 'attention' in p:
            return """Next steps to push past 82%:

1. ADD ATTENTION (Experiment 004):
   Self-attention over your 10 CNN windows will let the model 
   weight which 3-second segments are most informative.
   Expected gain: +2-4% over CNN-LSTM.

2. MULTI-SCALE CNN:
   Use filters of different sizes (3, 5, 7) in parallel.
   Captures both fine-grained and broad frequency patterns.

3. DATA AUGMENTATION:
   Add Gaussian noise to training epochs (σ = 0.01).
   EEG is inherently noisy — teaching robustness helps.

4. LEAVE-ONE-SUBJECT-OUT validation:
   Your current split may have subject leakage.
   True cross-subject performance is the publishable metric.

Start with attention — it's the most principled improvement
and directly builds toward the transformer architecture."""

        elif 'paper' in p or 'publish' in p or 'introduc' in p:
            return """Sleep disorders affect over 70 million people worldwide, 
yet automated sleep staging remains challenging due to the 
complexity of EEG signals and significant inter-subject 
variability. Accurate sleep stage classification is 
fundamental to both clinical diagnosis and emerging 
applications in brain-computer interfaces (BCI).

Recent deep learning approaches have demonstrated 
promising results on standardised datasets such as 
Sleep-EDF, with LSTM-based models achieving 74-82% 
accuracy on single-channel EEG. However, existing methods 
either process raw temporal sequences (losing frequency 
structure) or convert signals to spectrograms (losing 
temporal dynamics).

We propose a CNN-LSTM hybrid architecture that addresses 
both limitations. Our approach extracts local frequency 
features from 3-second windows using convolutional layers, 
then models temporal transitions between windows using a 
bidirectional LSTM. Trained on the Sleep-EDF Cassette 
dataset (20 subjects), our system achieves 80.14% overall 
accuracy with REM F1-score of 0.81 — improvements of 3.29% 
and 0.07 respectively over a pure LSTM baseline.

Our primary contribution is the demonstration that 
combining spatial feature extraction with temporal 
sequence modelling significantly outperforms either 
approach alone, with the largest gains observed in the 
notoriously difficult N1 stage (+34% relative F1 improvement).

This architecture is deployed as the classification backbone 
of a closed-loop BCI system for automated lucid dream 
induction, demonstrating practical applicability beyond 
academic benchmarking."""

        elif 'roadmap' in p or 'month' in p:
            return """6-Month Project 07 Roadmap:

MONTH 1 — Experiment 004 + Validation
  Goal: Transformer beats CNN-LSTM (target 82%+)
  Tasks: Train EEGTransformerClassifier,
         implement leave-one-subject-out validation,
         attention weight visualisation
  Deliverable: experiment_004_transformer notebook
  Risk: transformer may need more data than LSTM

MONTH 2 — Real Hardware
  Goal: Run pipeline on real EEG device
  Tasks: Order Muse S headband (~₹30,000),
         build real-time stream reader,
         integrate with detect_rem.py
  Deliverable: realtime_eeg_stream.py
  Risk: Muse API changes, latency issues

MONTH 3 — Self-Experiment Protocol
  Goal: First sleep experiment on own data
  Tasks: Personal calibration, logging protocol,
         experiment ethics documentation,
         record 5 nights of sleep data
  Deliverable: personal_sleep_dataset/

MONTH 4 — Neural Decoding Exploration
  Goal: Understand what dream imagery looks like in EEG
  Tasks: Read MinD-Vis and reconstruction papers,
         experiment with spectrogram → image pipeline,
         explore fNIRS as complement to EEG
  Deliverable: literature_review.md in PROJECT-07

MONTH 5 — Paper Writing
  Goal: Submit to a workshop or conference
  Tasks: Write full paper (8 pages IEEE format),
         create figures from experiment results,
         target: IEEE SMC or NeurIPS workshop
  Deliverable: paper_draft_v1.pdf

MONTH 6 — LUCID v0.2
  Goal: Working prototype on own sleep data
  Tasks: Integrate all components end-to-end,
         document protocol for others to replicate,
         record demo video
  Deliverable: LUCID v0.2 release on GitHub"""

        else:
            return (f"[Mock response for: '{prompt[:60]}...']\n"
                    "In production this would call claude-sonnet-4-6.\n"
                    "Add your Anthropic API key to get real responses.\n"
                    "Get one free at: console.anthropic.com")


# ── Conversation manager ───────────────────────────────────
class ResearchChat:
    def __init__(self):
        self.llm     = MockLLMClient()
        self.history = []

    def chat(self, message, max_tokens=500):
        self.history.append({
            'role': 'user',
            'content': message
        })
        reply = self.llm.ask(message)
        self.history.append({
            'role': 'assistant',
            'content': reply
        })
        return reply

    def clear(self):
        self.history = []


print(" Setup complete — mock LLM ready")
print("Replace MockLLMClient with anthropic.Anthropic()")
print("when you have an API key\n")

 Setup complete — mock LLM ready
Replace MockLLMClient with anthropic.Anthropic()
when you have an API key



In [9]:
# ── Test 1: Research questions ────────────────────────────
chat = ResearchChat()

questions = [
    "Why is N1 sleep stage the hardest to classify?",
    "What EEG features are most discriminative for REM?",
    "What should I try next to push accuracy above 82%?"
]

print("=== Sleep Research Assistant ===\n")
for q in questions:
    print(f"Q: {q}\n")
    reply = chat.chat(q)
    print(f"A: {textwrap.fill(reply, 70)}\n")
    print("-" * 60 + "\n")


# ── Test 2: Experiment analyser ───────────────────────────
exp_results = {
    "accuracy": 0.8014,
    "rem_f1":   0.81,
    "n1_f1":    0.47,
    "vs_lstm":  "+3.29%"
}

prompt = (f"Analyse these results and tell me "
          f"if this is publishable: "
          f"{json.dumps(exp_results)}")

print("=== Experiment Analysis ===\n")
analysis = chat.chat(prompt)
print(textwrap.fill(analysis, 70))

print("\n\n=== Paper Introduction Draft ===\n")
intro = chat.chat("Write a paper introduction for "
                  "my CNN-LSTM sleep staging research")
print(intro)

print("\n\n=== 6-Month Roadmap ===\n")
roadmap = chat.chat("Generate a 6-month roadmap for "
                    "Project 07 with monthly milestones")
print(roadmap)

MockLLMClient ready
(Replace with anthropic.Anthropic() when you have an API key)

=== Sleep Research Assistant ===

Q: Why is N1 sleep stage the hardest to classify?

A: N1 is the hardest sleep stage to classify for three reasons:  1.
BREVITY: N1 typically lasts only 1-7 minutes, giving the
classifier very few training examples relative to N2 (45-55%).  2.
SPECTRAL OVERLAP: N1 EEG shows mixed frequencies — alpha     waves
(8-13Hz) from wake begin to slow into theta (4-8Hz).     This overlap
makes it ambiguous.  3. INTER-SUBJECT VARIABILITY: N1 signatures vary
more between     people than any other stage, making generalisation
harder.  For your CNN-LSTM (N1 F1=0.47), the CNN windows help because
they capture the TRANSITION from alpha to theta across the  30-second
epoch — exactly what distinguishes N1 from wake.

------------------------------------------------------------

Q: What EEG features are most discriminative for REM?

A: Most discriminative EEG features for REM vs NREM:  1. T